[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anandjha90/Advanced-Python-for-GenAI-Prep-Training/blob/main/notebooks/03_Linear_Algebra.ipynb)

# Module 3: Linear Algebra for ML and GenAI (20 min)

Linear algebra is the language of data and neural networks. A dataset is a matrix, a word or product is represented as a vector (embedding), a neural network layer is a matrix multiplication, and semantic search is a dot product.

## 3.1 Scalars, vectors, matrices and tensors

| Object | Dimensions | NumPy shape | Deep learning use |
|---|---|---|---|
| Scalar | 0-D | () | A single loss value |
| Vector | 1-D | (n,) | One customer's features, a bias vector, a word embedding |
| Matrix | 2-D | (rows, cols) | A dataset (samples x features), a layer's weight matrix |
| Tensor | 3-D and more | (batch, channels, height, width) | A batch of images or a batch of token sequences |


The **shape** of an array is the most important thing to check when debugging ML code. Most deep learning errors are shape mismatches.

In [ ]:
import numpy as np

scalar = np.array(5)                                   # 0-D: e.g. one loss value
vector = np.array([1, 2, 3])                           # 1-D: e.g. one customer's features
matrix = np.array([[1, 2, 3], [4, 5, 6]])              # 2-D: e.g. a table of customers x features
tensor = np.zeros((32, 3, 224, 224))                   # 4-D: a batch of 32 RGB images, 224x224

for name, obj in [("Scalar", scalar), ("Vector", vector), ("Matrix", matrix), ("Tensor", tensor)]:
    print(f"{name:7}: ndim={obj.ndim}, shape={obj.shape}")

# Real-world example: 4 customers described by [age, orders, avg_spend]
customers = np.array([[25, 3, 1200],
                      [41, 12, 3400],
                      [33, 7, 2100],
                      [19, 1, 600]])
print("\nCustomer matrix shape:", customers.shape, "-> 4 rows (samples) x 3 columns (features)")
print("Second customer vector:", customers[1])
print("All ages (column 0):   ", customers[:, 0])

# Reshape: the same data viewed with a different shape (common before feeding a model)
flat = customers.reshape(-1)          # -1 means "work it out for me"
print("Flattened:", flat, "shape", flat.shape)
print("Back to 2 x 6:\n", flat.reshape(2, 6))

## 3.2 Vector operations, norms, dot product and cosine similarity

- **Addition / subtraction**: element-wise. **Scalar multiplication** stretches a vector without changing its direction.
- **Norm** = length of a vector. L1 (Manhattan) = sum of absolute values; L2 (Euclidean) = square root of sum of squares; Infinity norm = largest absolute value. L1 and L2 norms are also used for regularisation (Lasso and Ridge / weight decay).
- **Dot product** u . v = sum of u_i x v_i. It measures how much two vectors point in the same direction. A neuron computes a dot product between inputs and weights.
- **Cosine similarity** = (u . v) / (||u|| x ||v||). It ignores length and measures only direction: 1 = same direction, 0 = unrelated (orthogonal), -1 = opposite.

Worked example from the primer: u = [3, 4], v = [6, 0]. u . v = 18, ||u|| = 5, ||v|| = 6, so cosine similarity = 18 / 30 = **0.6**.

![Figure 5: Cosine similarity compares directions of embedding vectors (left); each entry of a matrix product is a row-column dot product (right).](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d05_vectors_matrix.png)
*Figure 5: Cosine similarity compares directions of embedding vectors (left); each entry of a matrix product is a row-column dot product (right).*


> **Real-world example**
>
> Embedding models turn every product, document or sentence into a vector of hundreds of numbers. Similar items get vectors that point in similar directions. "Customers who viewed earbuds also viewed..." and every vector database (Pinecone, FAISS, Snowflake Cortex Search) rank results by cosine similarity.

In [ ]:
import numpy as np

u = np.array([3, 4])
v = np.array([6, 0])

print("u + v =", u + v)                 # [9 4]
print("u - v =", u - v)                 # [-3 4]
print("2u    =", 2 * u)                 # [6 8]

# Norms (vector length)
x = np.array([3, -4, 12])
print("\nL1 norm  :", np.linalg.norm(x, 1))        # |3|+|-4|+|12| = 19
print("L2 norm  :", np.linalg.norm(x))           # sqrt(9+16+144) = 13
print("Inf norm :", np.linalg.norm(x, np.inf))   # max |x_i| = 12

# Dot product (example from the primer)
x = np.array([2, -1, 5])
w = np.array([1, 3, -2])
print("\nx . w =", np.dot(x, w), "| same as x @ w =", x @ w)    # -11

# Cosine similarity (example from the primer): u=[3,4], v=[6,0] -> 0.6
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print("cos(u, v) =", round(cosine_similarity(u, v), 2))

# Real-world example: toy 4-D "embeddings" of products
# dims ~ [electronics-ness, fashion-ness, sports-ness, price level]
emb = {
    "wireless earbuds": np.array([0.90, 0.05, 0.20, 0.40]),
    "bluetooth speaker": np.array([0.85, 0.02, 0.10, 0.45]),
    "running shoes":     np.array([0.05, 0.60, 0.80, 0.50]),
    "yoga mat":          np.array([0.02, 0.10, 0.90, 0.20]),
    "denim jeans":       np.array([0.01, 0.95, 0.05, 0.40]),
}
query = "wireless earbuds"
scores = {name: cosine_similarity(emb[query], vec) for name, vec in emb.items() if name != query}
print(f"\nProducts most similar to '{query}':")
for name, s in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {name:18} {s:.3f}")

## 3.3 Matrix operations and solving linear systems

- **Matrix multiplication** C = AB: entry C[i, j] is the dot product of row i of A and column j of B. The inner dimensions must match: (m x n)(n x p) gives (m x p). Order matters: AB is usually not equal to BA.
- **Transpose** A^T flips rows and columns. It is used in backpropagation.
- **Inverse** A^-1 "undoes" A, so A A^-1 = I (identity). Only square matrices with a non-zero **determinant** have an inverse (a zero determinant means a singular matrix).
- **System of equations** Ax = b: `np.linalg.solve` finds x. Deep learning avoids direct inversion for huge matrices and uses gradient descent instead.
- **Broadcasting**: NumPy automatically stretches a smaller array (e.g. a bias vector) to match a larger one.

> **Real-world example**
>
> Three customers bought different combinations of T-shirts, jeans and caps, and we only know their bill totals. Writing quantities as a matrix and bills as a vector, solving Ax = b recovers the unit price of each product. Matrix multiplication then prices many new baskets in one step.

In [ ]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[2, 0], [1, 2]])
print("A @ B =\n", A @ B)                 # [[4 4] [10 8]]  (example from the primer)
print("B @ A =\n", B @ A, "\n(order matters: AB != BA)")
print("A transpose =\n", A.T)
print("det(A) =", round(np.linalg.det(A), 2))
A_inv = np.linalg.inv(A)
print("A inverse =\n", A_inv)
print("A @ A_inv =\n", np.round(A @ A_inv, 10), "(identity matrix)")

# Real-world example: find unit prices from three customers' bills (Ax = b)
# Customer 1 bought 2 tees, 1 jeans, 1 cap  -> Rs 3,096
# Customer 2 bought 1 tee, 2 jeans, 0 caps  -> Rs 3,497
# Customer 3 bought 1 tee, 0 jeans, 3 caps  -> Rs 2,296
quantities = np.array([[2, 1, 1],
                       [1, 2, 0],
                       [1, 0, 3]])
bills = np.array([3096, 3497, 2296])
unit_prices = np.linalg.solve(quantities, bills)
print("\nUnit prices [tee, jeans, cap]:", unit_prices.round(2))

# Matrix multiplication for many baskets at once
baskets = np.array([[1, 1, 0], [0, 1, 1], [4, 0, 0]])   # 3 new baskets
print("Basket totals:", baskets @ unit_prices)

# Broadcasting: apply a different discount to each product column in one step
discount = np.array([0.10, 0.20, 0.00])
print("Discounted prices per basket:\n", baskets * unit_prices * (1 - discount))

## 3.4 Eigenvalues, SVD and PCA

- An **eigenvector** v of a square matrix A keeps its direction when multiplied by A; it is only stretched by the **eigenvalue** lambda: **Av = lambda v**.
- **Singular Value Decomposition (SVD)** factors any matrix into rotation x scaling x rotation: **A = U Sigma V^T**. Keeping only the largest singular values gives the best low-rank approximation of the matrix. This is used for recommendations, compression and LoRA fine-tuning of LLMs (low-rank weight updates).
- **Principal Component Analysis (PCA)** uses eigenvectors of the covariance matrix to find the directions of maximum variance and compresses many features into a few.

> **Real-world example**
>
> A users x products rating matrix has many empty cells. SVD finds hidden "taste" patterns (for example, an electronics lover vs a fitness lover) and fills in predicted ratings, which drive "Recommended for you". PCA is used to visualise high-dimensional embeddings in 2-D.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1) Eigenvalues and eigenvectors: A v = lambda v
A = np.array([[2, 1], [1, 2]])
eigenvalues, eigenvectors = np.linalg.eigh(A)    # eigh = for symmetric matrices
print("Eigenvalues:", eigenvalues)
v = eigenvectors[:, 1]                            # eigenvector for the largest eigenvalue
print("A @ v      :", A @ v)
print("lambda * v :", eigenvalues[1] * v, "(same direction, only scaled)")

# 2) SVD for recommendations: users x products rating matrix (0 = not rated)
ratings = np.array([[5, 4, 0, 1, 0],
                    [4, 5, 1, 0, 1],
                    [1, 0, 5, 4, 5],
                    [0, 1, 4, 5, 4],
                    [5, 5, 0, 0, 1]], dtype=float)
U, S, Vt = np.linalg.svd(ratings)
print("\nSingular values:", S.round(2))
k = 2                                             # keep the 2 strongest "taste" patterns
approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
print("Rank-2 reconstruction (predicted ratings):\n", approx.round(1))
print("Predicted score of user 0 for product 2:", approx[0, 2].round(2), "(low -> do not recommend)")

# 3) PCA on a real dataset: 13 chemical features of wines -> 2 dimensions
wine = load_wine()
X = StandardScaler().fit_transform(wine.data)
pca = PCA(n_components=2)
X2 = pca.fit_transform(X)
print("\nExplained variance by PC1, PC2:", pca.explained_variance_ratio_.round(3),
      "| total:", pca.explained_variance_ratio_.sum().round(3))

for label in np.unique(wine.target):
    plt.scatter(X2[wine.target == label, 0], X2[wine.target == label, 1],
                label=wine.target_names[label], alpha=0.7)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("PCA: 13 features compressed to 2")
plt.show()

## 3.5 Linear algebra inside a neural network

A dense (fully connected) layer does three things: rotate/scale the input with a weight matrix W, shift it with a bias vector b, and bend it with a non-linear activation:

**Forward pass: Z = XW + b, then A = activation(Z)**

With a batch of 2 samples and 3 features (X is 2 x 3) and a layer of 4 neurons (W is 3 x 4), the output is 2 x 4. During training, the transpose W^T carries error gradients backward.

In [ ]:
import numpy as np

# One dense neural-network layer is just linear algebra: Z = XW + b, A = ReLU(Z)
rng = np.random.default_rng(0)
X = np.array([[0.5, 1.2, -0.3],          # 2 samples (batch), 3 input features each
              [1.0, -0.7, 2.0]])
W = rng.normal(0, 0.5, size=(3, 4))      # 3 inputs -> 4 neurons
b = np.zeros(4)                          # one bias per neuron

Z = X @ W + b                            # (2x3) @ (3x4) + (4,) -> (2x4), bias is broadcast
A = np.maximum(0, Z)                     # ReLU activation
print("X shape", X.shape, "| W shape", W.shape, "| output shape", A.shape)
print("Z (before activation):\n", Z.round(3))
print("A (after ReLU):\n", A.round(3))

## 3.6 Module 3 practical: mini semantic product search

This is the retrieval step of a RAG system built only with linear algebra. Each product description becomes a TF-IDF vector (a row of a matrix). The user's query becomes a vector in the same space, and products are ranked by cosine similarity. In production, TF-IDF is replaced by neural embeddings, but the maths is identical.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# PRACTICAL: mini product search engine = the "retrieval" step of RAG
catalog = pd.DataFrame({
    "product": ["Wireless Earbuds", "Bluetooth Speaker", "Running Shoes", "Yoga Mat",
                "Denim Jeans", "Air Fryer", "Mixer Grinder", "Face Serum", "Sunscreen SPF50",
                "Smart Watch"],
    "description": [
        "bluetooth wireless earbuds with noise cancellation and long battery life",
        "portable bluetooth speaker with deep bass and waterproof body",
        "lightweight running shoes with cushioned sole for jogging and gym",
        "non slip yoga mat for home workout, gym and stretching",
        "slim fit blue denim jeans for men, stretchable cotton fabric",
        "air fryer for oil free cooking, crispy fries and snacks",
        "powerful mixer grinder with three jars for chutney and juice",
        "vitamin c face serum for glowing skin and dark spots",
        "sunscreen spf 50 for skin protection from sun, non greasy",
        "smart watch with heart rate, step tracking and bluetooth calling",
    ],
})

# Step 1: turn every description into a vector (a matrix: products x vocabulary)
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(catalog["description"])
print("Document-term matrix shape:", doc_vectors.shape)

# Step 2: turn the user's query into a vector and rank products by cosine similarity
def search(query, top_k=3):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, doc_vectors).ravel()
    best = [i for i in np.argsort(scores)[::-1][:top_k] if scores[i] > 0]
    return catalog.iloc[best].assign(score=scores[best].round(3))[["product", "score"]]

for q in ["something for my gym workout", "bluetooth music", "skin care for sun"]:
    print(f"\nQuery: '{q}'")
    print(search(q).to_string(index=False))

# Step 3: verify the library with our own NumPy cosine formula
q = vectorizer.transform(["bluetooth music"]).toarray()[0]
d = doc_vectors[1].toarray()[0]
manual = q @ d / (np.linalg.norm(q) * np.linalg.norm(d))
print(f"\nManual cosine for 'Bluetooth Speaker' = {manual:.3f}")

> **Discussion**
>
> TF-IDF only matches words that literally appear. The query "something for my gym workout" finds the yoga mat and running shoes because "gym" and "workout" appear in their descriptions, but a query like "fitness gear" would find nothing. Neural embeddings (Module 5) understand meaning, so "fitness" and "gym" end up close together.